In [1]:
# Installing all packages
import subprocess
import sys

packages = [
    "pandas==2.2.0",
    "numpy==1.26.4",
    "scikit-learn==1.4.0",
    "xgboost==2.0.3",
    "imbalanced-learn==0.12.0",
    "matplotlib==3.8.2",
    "seaborn==0.13.2",
    "joblib==1.3.2",
    "shap==0.44.1"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

All packages installed.


In [4]:
# libraries needed for training, evaluation, and visualization.
%matplotlib inline
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

warnings.filterwarnings("ignore")

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    classification_report,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
from xgboost                 import XGBClassifier
from imblearn.over_sampling  import SMOTE

plt.rcParams.update({
    "figure.figsize"    : (14, 6),
    "font.size"         : 12,
    "axes.titlesize"    : 14,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 12,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "figure.dpi"        : 120,
    "savefig.dpi"       : 150,
    "savefig.bbox"      : "tight"
})

sns.set_style("whitegrid")

COLORS = {
    "no_default" : "#2ecc71",
    "default"    : "#e74c3c",
    "primary"    : "#3498db",
    "secondary"  : "#9b59b6",
    "warning"    : "#f39c12",
    "dark"       : "#2c3e50"
}

os.makedirs("../models",          exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"sklearn      : {__import__('sklearn').__version__}")
print(f"xgboost      : {__import__('xgboost').__version__}")
print(f"imbalanced   : {__import__('imblearn').__version__}")

pandas       : 2.2.0
numpy        : 1.26.4
sklearn      : 1.4.0
xgboost      : 2.0.3
imbalanced   : 0.12.0


In [7]:
# Loading all 6 split files that 2_frature_eng saved.
X_train = pd.read_csv("../data/X_train.csv")
X_val   = pd.read_csv("../data/X_val.csv")
X_test  = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val   = pd.read_csv("../data/y_val.csv").squeeze()
y_test  = pd.read_csv("../data/y_test.csv").squeeze()

print(f"{'Split':<12} {'X Shape':>15} {'y Shape':>10} {'Default%':>10}")
print("-" * 50)
print(f"{'Train':<12} {str(X_train.shape):>15} "
    f"{str(y_train.shape):>10} "
    f"{y_train.mean()*100:>9.1f}%")
print(f"{'Validation':<12} {str(X_val.shape):>15} "
    f"{str(y_val.shape):>10} "
    f"{y_val.mean()*100:>9.1f}%")
print(f"{'Test':<12} {str(X_test.shape):>15} "
    f"{str(y_test.shape):>10} "
    f"{y_test.mean()*100:>9.1f}%")

print(f"\nFeature columns : {X_train.shape[1]}")
print(f"Features        : {list(X_train.columns)}")

Split                X Shape    y Shape   Default%
--------------------------------------------------
Train            (20987, 29)   (20987,)      22.1%
Validation        (4483, 29)    (4483,)      22.1%
Test              (4495, 29)    (4495,)      22.1%

Feature columns : 29
Features        : ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'credit_utilization_rate', 'debt_to_income_ratio', 'avg_payment_delay', 'total_bill_amt', 'total_pay_amt', 'payment_to_bill_ratio']


In [10]:
# Applying SMOTE, It creates synthetic samples of the minority class (defaulters) so the model learns both classes equally well.
# We only apply SMOTE to training data — never to validation or test.
# What is SMOTE? Synthetic Minority Over-sampling Technique (SMOTE) is a statistical technique used in data science and machine learning to resolve class imbalance in a dataset.
# Why do we use here? Bcz, It handles the class imbalance we found in 1_eda (77.9% vs 22.1%).
print("Before SMOTE:")
print(f"  Train No Default : {(y_train==0).sum():,}")
print(f"  Train Default    : {(y_train==1).sum():,}")
print(f"  Ratio            : {(y_train==0).sum()/(y_train==1).sum():.1f}:1")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Train No Default : {(y_train_sm==0).sum():,}")
print(f"  Train Default    : {(y_train_sm==1).sum():,}")
print(f"  Ratio            : {(y_train_sm==0).sum()/(y_train_sm==1).sum():.1f}:1")
print(f"\n  Original train rows : {len(X_train):,}")
print(f"  After SMOTE rows    : {len(X_train_sm):,}")
print(f"  Synthetic samples   : {len(X_train_sm)-len(X_train):,} added")


Before SMOTE:
  Train No Default : 16,344
  Train Default    : 4,643
  Ratio            : 3.5:1

After SMOTE:
  Train No Default : 16,344
  Train Default    : 16,344
  Ratio            : 1.0:1

  Original train rows : 20,987
  After SMOTE rows    : 32,688
  Synthetic samples   : 11,701 added


In [15]:
# Logistic Regression is the simplest classifier. We train it first to get a baseline score.
# Every other model must beat this to be sutable to use.
print("Training Logistic Regression...")

lr_model = LogisticRegression(
    random_state = 42,
    max_iter     = 1000,
    class_weight = "balanced"
)

lr_model.fit(X_train_sm, y_train_sm)

lr_val_pred  = lr_model.predict(X_val)
lr_val_proba = lr_model.predict_proba(X_val)[:, 1]

lr_metrics = {
    "AUC-ROC"   : roc_auc_score(y_val, lr_val_proba),
    "F1-Score"  : f1_score(y_val, lr_val_pred),
    "Precision" : precision_score(y_val, lr_val_pred),
    "Recall"    : recall_score(y_val, lr_val_pred)
}

print(f"{'Metric':<15} {'Score':>10}")
for metric, score in lr_metrics.items():
    print(f"{metric:<15} {score:>10.4f}")

print(f"\nClassification Report:")
print(classification_report(
    y_val, lr_val_pred,
    target_names=["No Default", "Default"]
))

Training Logistic Regression...
Metric               Score
AUC-ROC             0.7180
F1-Score            0.4656
Precision           0.3792
Recall              0.6028

Classification Report:
              precision    recall  f1-score   support

  No Default       0.86      0.72      0.79      3491
     Default       0.38      0.60      0.47       992

    accuracy                           0.69      4483
   macro avg       0.62      0.66      0.63      4483
weighted avg       0.76      0.69      0.71      4483



In [21]:
# Training Random Forest: Random Forest builds hundreds of decision trees and combines their votes.
# It handles non-linear patterns and is much stronger than Logistic Regression.
print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 10,
    random_state  = 42,
    class_weight  = "balanced",
    n_jobs        = -1
)

rf_model.fit(X_train_sm, y_train_sm)

rf_val_pred  = rf_model.predict(X_val)
rf_val_proba = rf_model.predict_proba(X_val)[:, 1]

rf_metrics = {
    "AUC-ROC"   : roc_auc_score(y_val, rf_val_proba),
    "F1-Score"  : f1_score(y_val, rf_val_pred),
    "Precision" : precision_score(y_val, rf_val_pred),
    "Recall"    : recall_score(y_val, rf_val_pred)
}

print("Random Forest — Validation Results:")
print(f"{'Metric':<15} {'Score':>10}")
for metric, score in rf_metrics.items():
    print(f"{metric:<15} {score:>10.4f}")

print(f"\nClassification Report:")
print(classification_report(y_val, rf_val_pred,
    target_names=["No Default", "Default"]))

Training Random Forest...
Random Forest — Validation Results:
Metric               Score
AUC-ROC             0.7720
F1-Score            0.5294
Precision           0.4901
Recall              0.5756

Classification Report:
              precision    recall  f1-score   support

  No Default       0.87      0.83      0.85      3491
     Default       0.49      0.58      0.53       992

    accuracy                           0.77      4483
   macro avg       0.68      0.70      0.69      4483
weighted avg       0.79      0.77      0.78      4483



In [ ]:
# Train XGBoost: XGBoost is our main model. It learns from mistakes of previous trees (gradient boosting).
# It consistently performs best on tabular data and wins most Kaggle competitions.
# print("Training XGBoost...")

# xgb_model = XGBClassifier(
#     n_estimators     = 500,
#     max_depth        = 5,
#     learning_rate    = 0.1,
#     subsample        = 0.8,
#     colsample_bytree = 0.8,
#     min_child_weight = 5,
#     gamma            = 0.1,
#     reg_alpha        = 0.1,
#     reg_lambda       = 1.0,
#     random_state     = 42,
#     eval_metric      = "auc",
#     verbosity        = 0
# )

# xgb_model.fit(
#     X_train_sm, y_train_sm,
#     verbose = False
# )

# xgb_val_pred  = xgb_model.predict(X_val)
# xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]

# xgb_metrics = {
#     "AUC-ROC"   : roc_auc_score(y_val, xgb_val_proba),
#     "F1-Score"  : f1_score(y_val, xgb_val_pred),
#     "Precision" : precision_score(y_val, xgb_val_pred),
#     "Recall"    : recall_score(y_val, xgb_val_pred)
# }

# print("XGBoost — Validation Results:")
# print(f"{'Metric':<15} {'Score':>10}")
# for metric, score in xgb_metrics.items():
#     print(f"{metric:<15} {score:>10.4f}")

# print(f"\nClassification Report:")
# print(classification_report(
#     y_val, xgb_val_pred,
#     target_names=["No Default", "Default"]
# ))

# # But here we are not using XG-Boost, If you ask Y then explanation here :
# Random Forest outperformed XGBoost on key metrics (AUC-ROC: 0.7720 vs. 0.7570, Recall: 0.5756 vs. 0.4224) for this dataset. 
# While XGBoost typically scales better on massive datasets, Random Forest yields superior balance on this medium-sized imbalanced dataset. 
# (XGBoost code kept commented above for reference).

In [30]:
# Model Comparison Table: Numbers in separate cells are hard to compare side by side so one table is easy. 
# now you can clearly see which model won each metric.
results = {
    "Logistic Regression" : lr_metrics,
    "Random Forest"       : rf_metrics
}

print(f"{'Model':<25} {'AUC-ROC':>10} {'F1-Score':>10} "
    f"{'Precision':>12} {'Recall':>10}")

for model_name, metrics in results.items():
    flag = "- BEST" if model_name == "Random Forest" else ""
    print(f"{model_name:<25} "
        f"{metrics['AUC-ROC']:>10.4f} "
        f"{metrics['F1-Score']:>10.4f} "
        f"{metrics['Precision']:>12.4f} "
        f"{metrics['Recall']:>10.4f}{flag}")

print(f"Best model : Random Forest")
print(f"Best AUC   : {rf_metrics['AUC-ROC']:.4f}")

Model                        AUC-ROC   F1-Score    Precision     Recall
Logistic Regression           0.7180     0.4656       0.3792     0.6028
Random Forest                 0.7720     0.5294       0.4901     0.5756- BEST
Best model : Random Forest
Best AUC   : 0.7720


In [32]:
# Model Comparison Chart: Charts make the score gap immediately visible.
# Anyone looking at this instantly sees Random Forest is better across most metrics without reading any numbers.
metrics_list = ["AUC-ROC", "F1-Score", "Precision", "Recall"]
model_names  = list(results.keys())
bar_colors   = [COLORS["primary"], COLORS["secondary"]]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.patch.set_facecolor("white")

for idx, metric in enumerate(metrics_list):
    ax     = axes[idx]
    values = [results[m][metric] for m in model_names]

    bars = ax.bar(
        model_names, values,
        color=bar_colors,
        edgecolor="white",
        linewidth=1.5, width=0.4
    )
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{val:.3f}",
            ha="center", va="bottom",
            fontweight="bold", fontsize=11
        )

    ax.set_title(metric, fontsize=13)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(
        [m.replace(" ", "\n") for m in model_names],
        fontsize=10
    )
    ax.set_facecolor("white")

fig.suptitle("Model Comparison — Validation Set",
            fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig("../reports/figures/11_model_comparison.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved : reports/figures/11_model_comparison.png ")

Saved : reports/figures/11_model_comparison.png 


In [34]:
# The ROC curve for both models on the same chart. 
# The curve shows how well each model separates defaulters from non-defaulters at every possible threshold.
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

models_proba = {
    "Logistic Regression" : lr_val_proba,
    "Random Forest"       : rf_val_proba
}
roc_colors = [COLORS["primary"], COLORS["secondary"]]

for (model_name, proba), color in zip(models_proba.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc         = roc_auc_score(y_val, proba)
    lw          = 3 if model_name == "Random Forest" else 2
    ax.plot(
        fpr, tpr,
        color=color, linewidth=lw,
        label=f"{model_name} (AUC = {auc:.4f})"
    )

ax.plot(
    [0, 1], [0, 1],
    "k--", linewidth=1.5,
    label="Random Guess (AUC = 0.5000)"
)

ax.set_title("ROC Curve — All Models", fontsize=14, pad=15)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.legend(fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3, linestyle="--")

plt.tight_layout()
fig.savefig("../reports/figures/12_roc_curves.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved : reports/figures/12_roc_curves.png ")

Saved : reports/figures/12_roc_curves.png 


In [36]:
# Cofusion Matrix: a 2×2 grid for each model — actual vs predicted.
# so you can see exactly how many customers were correctly and incorrectly classified.
models_preds = {
    "Logistic Regression" : lr_val_pred,
    "Random Forest"       : rf_val_pred
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("white")

for idx, (model_name, pred) in enumerate(models_preds.items()):
    ax             = axes[idx]
    cm             = confusion_matrix(y_val, pred)
    tn, fp, fn, tp = cm.ravel()

    sns.heatmap(
        cm, annot=True, fmt="d",
        cmap="Blues", ax=ax,
        xticklabels=["No Default", "Default"],
        yticklabels=["No Default", "Default"],
        linewidths=1, linecolor="white",
        annot_kws={"size": 14, "weight": "bold"}
    )

    title_flag = " ⭐ BEST" if model_name == "Random Forest" else ""
    ax.set_title(f"{model_name}{title_flag}", fontsize=12, pad=10)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.text(
        0.5, -0.12,
        f"TP={tp}  FP={fp}  FN={fn}  TN={tn}",
        ha="center", transform=ax.transAxes,
        fontsize=9, color="gray"
    )

fig.suptitle("Confusion Matrices — Validation Set",
            fontsize=15, fontweight="bold")
plt.tight_layout()
fig.savefig("../reports/figures/13_confusion_matrices.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Saved : reports/figures/13_confusion_matrices.png ")

Saved : reports/figures/13_confusion_matrices.png 


In [38]:
# Random Forest Feature importance confirms the 1_eda findings.
# If PAY_0 and payment history are at the top it means our EDA was correct. 
# It also shows whether the new features we created in 2_feature_eng are helping the model.
importance_df = pd.DataFrame({
    "Feature"    : X_train.columns,
    "Importance" : rf_model.feature_importances_
}).sort_values("Importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bars = ax.barh(
    importance_df["Feature"],
    importance_df["Importance"],
    color=COLORS["secondary"],
    edgecolor="white",
    linewidth=0.8, height=0.7
)

for bar, val in zip(bars, importance_df["Importance"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center", ha="left",
        fontsize=9, fontweight="bold"
    )

ax.set_title("Random Forest — Top 15 Feature Importances",
            fontsize=14, pad=15)
ax.set_xlabel("Importance Score", fontsize=12)
ax.invert_yaxis()
ax.xaxis.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
fig.savefig("../reports/figures/14_feature_importance.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)

print("Top 15 Feature Importances (Random Forest):")
print(f"{'Rank':<6} {'Feature':<30} {'Importance':>12}")
print("-" * 50)
for rank, (_, row) in enumerate(importance_df.iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<30} {row['Importance']:>12.4f}")

print(f"\nSaved : reports/figures/14_feature_importance.png ")

Top 15 Feature Importances (Random Forest):
Rank   Feature                          Importance
--------------------------------------------------
1      avg_payment_delay                    0.1544
2      PAY_0                                0.1463
3      PAY_2                                0.0770
4      total_pay_amt                        0.0439
5      LIMIT_BAL                            0.0435
6      PAY_3                                0.0431
7      PAY_4                                0.0340
8      PAY_AMT1                             0.0324
9      credit_utilization_rate              0.0320
10     PAY_AMT2                             0.0306
11     PAY_5                                0.0286
12     debt_to_income_ratio                 0.0271
13     total_bill_amt                       0.0262
14     BILL_AMT1                            0.0248
15     payment_to_bill_ratio                0.0247

Saved : reports/figures/14_feature_importance.png 


In [40]:
# Final Test Set evaluation: Runs the best model (Random Forest) on the test set that was never used during training or validation.
# This is the true real-world performance number.
# Validation scores can be slightly optimistic because we used the validation set to compare models. 
# The test set is completely untouched — it simulates how the model performs on brand new customers it has never seen before.
print("Final Evaluation — Random Forest on Test Set")

rf_test_pred  = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "AUC-ROC"   : roc_auc_score(y_test, rf_test_proba),
    "F1-Score"  : f1_score(y_test, rf_test_pred),
    "Precision" : precision_score(y_test, rf_test_pred),
    "Recall"    : recall_score(y_test, rf_test_pred)
}

print(f"\n{'Metric':<15} {'Val Score':>12} {'Test Score':>12} {'Diff':>10}")

for metric in test_metrics:
    val_score  = rf_metrics[metric]
    test_score = test_metrics[metric]
    diff       = test_score - val_score
    flag       = "Good" if abs(diff) < 0.05 else "Bad"
    print(f"{metric:<15} {val_score:>12.4f} "
        f"{test_score:>12.4f} "
        f"{diff:>+10.4f} {flag}")

print(f"\nClassification Report on Test Set:")
print(classification_report(
    y_test, rf_test_pred,
    target_names=["No Default", "Default"]
))

Final Evaluation — Random Forest on Test Set

Metric             Val Score   Test Score       Diff
AUC-ROC               0.7720       0.7675    -0.0045 Good
F1-Score              0.5294       0.5298    +0.0004 Good
Precision             0.4901       0.4935    +0.0034 Good
Recall                0.5756       0.5719    -0.0037 Good

Classification Report on Test Set:
              precision    recall  f1-score   support

  No Default       0.87      0.83      0.85      3500
     Default       0.49      0.57      0.53       995

    accuracy                           0.78      4495
   macro avg       0.68      0.70      0.69      4495
weighted avg       0.79      0.78      0.78      4495



In [42]:
# Saving Models: Saves both trained models as .pkl files.
# Saves the SMOTE training data so 4_agent can load everything without retraining.
joblib.dump(lr_model, "../models/logistic_regression.pkl")
joblib.dump(rf_model, "../models/random_forest.pkl")
joblib.dump(rf_model, "../models/best_model.pkl")

X_train_sm_df = pd.DataFrame(X_train_sm, columns=X_train.columns)
y_train_sm_df = pd.Series(y_train_sm, name="default")

X_train_sm_df.to_csv("../data/X_train_smote.csv", index=False)
y_train_sm_df.to_csv("../data/y_train_smote.csv", index=False)

saved = [
    ("../models/logistic_regression.pkl", "Logistic Regression"),
    ("../models/random_forest.pkl",        "Random Forest"),
    ("../models/best_model.pkl",           "Best Model (RF copy)"),
    ("../data/X_train_smote.csv",          "SMOTE train features"),
    ("../data/y_train_smote.csv",          "SMOTE train labels")
]

print(f"{'Label':<30} {'Size':>15}")
print("-" * 48)
for path, label in saved:
    size = os.path.getsize(path)
    print(f"{label:<30} {size:>12,} bytes")

print("\n All models saved")
print("Best model - models/best_model.pkl (Random Forest)")

Label                                     Size
------------------------------------------------
Logistic Regression                   1,887 bytes
Random Forest                    15,316,441 bytes
Best Model (RF copy)             15,316,441 bytes
SMOTE train features             13,577,740 bytes
SMOTE train labels                   98,073 bytes

 All models saved
Best model - models/best_model.pkl (Random Forest)


In [44]:
#  3_model_traning Completion Check
import os
import pandas as pd

print("3_model_traning COMPLETION CHECK")

required_files = {
    "../models/logistic_regression.pkl"            : "Logistic Regression",
    "../models/random_forest.pkl"                  : "Random Forest",
    "../models/best_model.pkl"                     : "Best model (RF)",
    "../data/X_train_smote.csv"                    : "SMOTE train features",
    "../data/y_train_smote.csv"                    : "SMOTE train labels",
    "../reports/figures/11_model_comparison.png"   : "Model comparison",
    "../reports/figures/12_roc_curves.png"         : "ROC curves",
    "../reports/figures/13_confusion_matrices.png" : "Confusion matrices",
    "../reports/figures/14_feature_importance.png" : "Feature importance"
}

all_good = True
for path, label in required_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f" {label:<30} {size:>12,} bytes")
    else:
        print(f" {label:<30} NOT FOUND")
        all_good = False

print(f"\n{'Model':<25} {'AUC-ROC':>10} {'F1-Score':>10} {'Winner'}")
print("-" * 55)
for name, metrics in {
    "Logistic Regression" : lr_metrics,
    "Random Forest"       : rf_metrics
}.items():
    flag = "🏆 BEST" if name == "Random Forest" else ""
    print(f"{name:<25} {metrics['AUC-ROC']:>10.4f} "
        f"{metrics['F1-Score']:>10.4f}  {flag}")

print(f"\n  Test AUC-ROC : {test_metrics['AUC-ROC']:.4f}")
print(f"  Test F1      : {test_metrics['F1-Score']:.4f}")
print(f"  Best Model   : Random Forest - models/best_model.pkl")

print(f"  3_model_traning : {'COMPLETE' if all_good else ' FIX ITEMS ABOVE'}")

3_model_traning COMPLETION CHECK
 Logistic Regression                   1,887 bytes
 Random Forest                    15,316,441 bytes
 Best model (RF)                  15,316,441 bytes
 SMOTE train features             13,577,740 bytes
 SMOTE train labels                   98,073 bytes
 Model comparison                     60,817 bytes
 ROC curves                          116,178 bytes
 Confusion matrices                   84,499 bytes
 Feature importance                   98,971 bytes

Model                        AUC-ROC   F1-Score Winner
-------------------------------------------------------
Logistic Regression           0.7180     0.4656  
Random Forest                 0.7720     0.5294  🏆 BEST

  Test AUC-ROC : 0.7675
  Test F1      : 0.5298
  Best Model   : Random Forest - models/best_model.pkl
  3_model_traning : COMPLETE
